# Cardiac Patient Monitoring System

## 02 — EDA and Statistics

## Objective

Explore the cleaned dataset using descriptive statistics, probability/statistics concepts, distributions, class balance, correlations, and visualizations.

## Imports and Load Clean Data

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

DATA_PATH = DATA_DIR / "cardio_train.csv"

OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)

CLEAN_PATH = DATA_DIR / "cardio_clean.csv"
if not CLEAN_PATH.exists():
    raise FileNotFoundError("Run 01_data_preparation.ipynb first.")
df = pd.read_csv(CLEAN_PATH)
df.head()


## Descriptive Statistics

In [ ]:
display(df.describe().T)


## Target Distribution

In [ ]:
class_counts = df["cardio"].value_counts().sort_index()
class_pct = df["cardio"].value_counts(normalize=True).sort_index() * 100

target_summary = pd.DataFrame({
    "count": class_counts,
    "percentage": class_pct.round(2)
})
display(target_summary)

plt.figure(figsize=(6,4))
class_counts.plot(kind="bar")
plt.title("Cardiovascular Disease Target Distribution")
plt.xlabel("cardio")
plt.ylabel("Number of records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "target_distribution.png", dpi=150)
plt.show()


## Numerical Feature Distributions

In [ ]:
numeric_features = ["age", "height", "weight", "ap_hi", "ap_lo"]
for col in numeric_features:
    plt.figure(figsize=(7,4))
    plt.hist(df[col], bins=40)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"distribution_{col}.png", dpi=150)
    plt.show()


## Group Comparison by Target

In [ ]:
grouped = df.groupby("cardio")[numeric_features].mean().T
display(grouped)


## Correlation Analysis

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10,8))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "correlation_matrix.png", dpi=150)
plt.show()

display(corr["cardio"].sort_values(ascending=False).to_frame("correlation_with_cardio"))


## Probability Perspective

The empirical probability of the positive target class can be estimated by its observed proportion in the dataset.

In [ ]:
p_positive = df["cardio"].mean()
p_negative = 1 - p_positive

print(f"P(cardio=1) ≈ {p_positive:.4f}")
print(f"P(cardio=0) ≈ {p_negative:.4f}")


## Potential Outliers

Boxplots are used to identify extreme observations visually. Detection does not automatically mean removal; an observation should only be removed when there is a defensible data-quality reason.

In [ ]:
for col in numeric_features:
    plt.figure(figsize=(7,3))
    plt.boxplot(df[col], vert=False)
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"boxplot_{col}.png", dpi=150)
    plt.show()


## Key EDA Questions to Discuss

1. Is the target reasonably balanced?
2. Which features show the strongest linear association with `cardio`?
3. Which numerical variables contain extreme values?
4. Do feature distributions differ between the two target classes?
5. Which findings should influence preprocessing or model selection?

## EDA Summary

The charts and statistics generated here provide the evidence used to justify the modeling and preprocessing decisions in the next notebooks.